# scRNA GEO: Playwright 수집 SRR vs v3 CSV 비교

- **v3**: `scRNA_seq_datasets_v3.csv` (GEO 행만) — `rawdata_name` 컬럼의 SRR
- **Playwright**: `output/geo_scrna_playwright_srr.csv` — GSE/GSM별로 수집한 `SRR_playwright`
- 두 소스를 GSE + Sample_accesion(GSM)으로 매칭한 뒤 SRR 일치 여부 분석

In [ ]:
import pandas as pd
import os

BASE = os.getcwd()
if not os.path.exists(os.path.join(BASE, 'scRNA_seq_datasets_v3.csv')):
    BASE = os.path.join(BASE, '01_playwright_geo_scrna')
V3_CSV = os.path.join(BASE, 'scRNA_seq_datasets_v3.csv')
PLAYWRIGHT_CSV = os.path.join(BASE, 'output', 'geo_scrna_playwright_srr.csv')
print('BASE', BASE)
print('v3 exists', os.path.exists(V3_CSV))
print('playwright exists', os.path.exists(PLAYWRIGHT_CSV))

## 1) v3 CSV에서 GEO 데이터셋 목록 및 행 로드

In [ ]:
with open(V3_CSV, 'r', encoding='utf-8', errors='replace') as f:
    lines = [l for l in f if l.strip() and not l.startswith('#')]
df_v3 = pd.read_csv(pd.io.common.StringIO(''.join(lines)))
geo_mask = df_v3['Accesion_url'].astype(str).str.contains('ncbi\.nlm\.nih\.gov/geo', na=False)
df_geo = df_v3.loc[geo_mask, ['Accesion', 'Sample_accesion', 'rawdata_name']].copy()
df_geo = df_geo.rename(columns={'Accesion': 'GSE', 'rawdata_name': 'SRR_v3'})
df_geo['SRR_v3'] = df_geo['SRR_v3'].astype(str).str.strip()

geo_datasets = df_geo['GSE'].unique().tolist()
print('GEO 관련 scRNA 데이터셋 (GSE) 목록:', len(geo_datasets))
print(geo_datasets)
print('\nGEO 행 수:', len(df_geo))
df_geo.head(10)

## 2) Playwright 수집 CSV 로드

In [ ]:
df_pw = pd.read_csv(PLAYWRIGHT_CSV)
df_pw['SRR_playwright'] = df_pw['SRR_playwright'].astype(str).str.strip()
df_pw.loc[df_pw['SRR_playwright'] == 'nan', 'SRR_playwright'] = ''
print('Playwright 수집 행 수:', len(df_pw))
print(df_pw.head(10).to_string())

## 3) GSE + Sample_accesion 기준 병합 및 SRR 비교

In [ ]:
merged = df_geo.merge(
    df_pw,
    on=['GSE', 'Sample_accesion'],
    how='left'
)
merged['SRR_playwright'] = merged['SRR_playwright'].fillna('')

def normalize_srr(s):
    if pd.isna(s) or s == '' or s == 'nan':
        return set()
    return set(x.strip() for x in str(s).replace(',', ' ').split() if x.strip().startswith('SRR'))

def compare_row(row):
    v3_set = normalize_srr(row['SRR_v3'])
    pw_set = normalize_srr(row['SRR_playwright'])
    if not v3_set and not pw_set:
        return 'both_empty'
    if not pw_set:
        return 'missing_in_playwright'
    if not v3_set:
        return 'only_in_playwright'
    if v3_set == pw_set:
        return 'exact_match'
    if v3_set & pw_set:
        return 'partial_match'
    return 'mismatch'

merged['compare'] = merged.apply(compare_row, axis=1)
print(merged['compare'].value_counts())
os.makedirs(os.path.join(BASE, 'output'), exist_ok=True)
merged_out = os.path.join(BASE, 'output', 'srr_comparison_merged.csv')
merged.to_csv(merged_out, index=False)
print('저장:', merged_out)
print(merged.head(15).to_string())

## 4) 요약 통계 및 SRR 불일치 샘플

In [ ]:
summary = merged['compare'].value_counts()
total = len(merged)
match = (merged['compare'] == 'exact_match').sum()
partial = (merged['compare'] == 'partial_match').sum()
missing = (merged['compare'] == 'missing_in_playwright').sum()
mismatch = (merged['compare'] == 'mismatch').sum()
both_empty = (merged['compare'] == 'both_empty').sum()

print('=== SRR 비교 요약 ===')
print(f'총 행: {total}')
print(f'exact_match (완전 일치): {match} ({100*match/total:.1f}%)')
print(f'partial_match (일부 일치): {partial}')
print(f'missing_in_playwright (v3에만 SRR 있음, Playwright 미수집): {missing}')
print(f'both_empty (v3·Playwright 둘 다 SRR 없음): {both_empty}')
print(f'mismatch (SRR 불일치): {mismatch}')
print()
print('=== mismatch 또는 partial_match 샘플 (처음 20건) ===')
disp = merged[merged['compare'].isin(['mismatch', 'partial_match'])][['GSE', 'Sample_accesion', 'SRR_v3', 'SRR_playwright', 'compare']]
display(disp.head(20))

In [ ]:
print('=== missing_in_playwright 샘플 (처음 15건) ===')
miss = merged[merged['compare'] == 'missing_in_playwright'][['GSE', 'Sample_accesion', 'SRR_v3', 'SRR_playwright']].head(15)
print(miss.to_string())

## 5) GSE별 일치율

In [ ]:
gse_stats = merged.groupby('GSE').agg(
    total=('Sample_accesion', 'count'),
    exact=('compare', lambda s: (s == 'exact_match').sum()),
    missing=('compare', lambda s: (s == 'missing_in_playwright').sum()),
    mismatch=('compare', lambda s: (s == 'mismatch').sum()),
)
gse_stats['match_pct'] = (gse_stats['exact'] / gse_stats['total'] * 100).round(1)
print(gse_stats.to_string())